# Stock Analysis Template

This notebook is designed to be run programmatically with **papermill**. It analyzes a single stock over a specified date range and produces:
- Price and volume charts
- Rolling volatility (Yang-Zhang estimator)
- Basic return statistics
- Data quality checks

**How this works**: The cell below tagged with `parameters` contains default values. When papermill runs this notebook, it injects a new cell with your custom values, overriding the defaults.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import scrapbook as sb

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('ggplot')
%matplotlib inline

In [ ]:
# PARAMETERS - These will be overwritten by papermill
# When you run this notebook manually, these defaults are used

ticker = 'AAPL'
start_date = '2024-01-01'
end_date = '2024-06-30'
db_path = 'data/data.db'
volatility_window = 20  # days for rolling volatility

## 1. Load Data from Database

In [ ]:
conn = sqlite3.connect(db_path)

query = f"""
SELECT ts, open, high, low, close, adj_close, volume
FROM ohlc
WHERE ticker = '{ticker}'
  AND ts >= '{start_date}'
  AND ts <= '{end_date}'
ORDER BY ts
"""

df = pd.read_sql(query, conn, parse_dates=['ts'], index_col='ts')
conn.close()

print(f"Loaded {len(df)} trading days for {ticker}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

## 2. Data Quality Checks

Before analysis, we verify the data meets basic quality standards.

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None - data is complete")

# Check OHLC integrity: high >= low, high >= open/close, low <= open/close
ohlc_violations = (
    (df['high'] < df['low']) |
    (df['high'] < df['open']) |
    (df['high'] < df['close']) |
    (df['low'] > df['open']) |
    (df['low'] > df['close'])
)
n_violations = ohlc_violations.sum()
print(f"\nOHLC integrity violations: {n_violations}")

# Check for negative prices or volume
negative_prices = (df[['open', 'high', 'low', 'close', 'adj_close']] <= 0).any().any()
negative_volume = (df['volume'] < 0).any()
print(f"Negative prices found: {negative_prices}")
print(f"Negative volume found: {negative_volume}")

# Store quality status
data_quality_ok = (n_violations == 0) and not negative_prices and not negative_volume

## 3. Calculate Features

We compute several useful features for analysis.

In [ ]:
# Daily returns (using adjusted close for accuracy)
df['daily_return'] = df['adj_close'].pct_change()

# Log returns (useful for many financial models)
df['log_return'] = np.log(df['adj_close'] / df['adj_close'].shift(1))

# Cumulative return from start of period
df['cumulative_return'] = (1 + df['daily_return']).cumprod() - 1

In [ ]:
def yang_zhang_volatility(df, window=20):
    """
    Yang-Zhang volatility estimator.
    
    This estimator uses OHLC data and is more efficient than close-to-close
    volatility. It accounts for overnight gaps and intraday price movements.
    
    Reference: Yang & Zhang (2000), "Drift Independent Volatility Estimation"
    """
    n = window
    k = 0.34 / (1.34 + (n + 1) / (n - 1))
    
    # Log price ratios
    log_ho = np.log(df['high'] / df['open'])
    log_lo = np.log(df['low'] / df['open'])
    log_co = np.log(df['close'] / df['open'])
    log_oc = np.log(df['open'] / df['close'].shift(1))  # overnight
    log_cc = np.log(df['close'] / df['close'].shift(1))  # close-to-close
    
    # Variance components
    var_overnight = (log_oc ** 2).rolling(n).sum() / (n - 1)
    var_close = (log_cc ** 2).rolling(n).sum() / (n - 1)
    
    # Rogers-Satchell variance (intraday)
    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)
    var_rs = rs.rolling(n).sum() / (n - 1)
    
    # Yang-Zhang variance
    var_yz = var_overnight + k * var_close + (1 - k) * var_rs
    
    # Annualized volatility (sqrt of variance * sqrt(252 trading days))
    return np.sqrt(var_yz) * np.sqrt(252)

df['volatility'] = yang_zhang_volatility(df, window=volatility_window)
print(f"Calculated {volatility_window}-day rolling volatility")

In [ ]:
# Intraday range as percentage of price
df['daily_range_pct'] = (df['high'] - df['low']) / df['close'] * 100

# Simple moving averages
df['sma_20'] = df['adj_close'].rolling(20).mean()
df['sma_50'] = df['adj_close'].rolling(50).mean()

# Volume moving average (for relative volume)
df['volume_sma_20'] = df['volume'].rolling(20).mean()
df['relative_volume'] = df['volume'] / df['volume_sma_20']

## 4. Summary Statistics

In [ ]:
# Calculate key statistics
stats = {
    'ticker': ticker,
    'start_date': start_date,
    'end_date': end_date,
    'trading_days': len(df),
    'start_price': df['adj_close'].iloc[0],
    'end_price': df['adj_close'].iloc[-1],
    'total_return_pct': df['cumulative_return'].iloc[-1] * 100,
    'avg_daily_return_pct': df['daily_return'].mean() * 100,
    'daily_return_std_pct': df['daily_return'].std() * 100,
    'avg_volatility': df['volatility'].mean(),
    'max_volatility': df['volatility'].max(),
    'min_volatility': df['volatility'].min(),
    'max_drawdown_pct': ((df['adj_close'] / df['adj_close'].cummax()) - 1).min() * 100,
    'avg_daily_volume': df['volume'].mean(),
    'max_single_day_return_pct': df['daily_return'].max() * 100,
    'min_single_day_return_pct': df['daily_return'].min() * 100,
}

# Display stats
print(f"\n=== {ticker} Summary: {start_date} to {end_date} ===")
print(f"Trading days: {stats['trading_days']}")
print(f"Price: ${stats['start_price']:.2f} -> ${stats['end_price']:.2f}")
print(f"Total return: {stats['total_return_pct']:.2f}%")
print(f"Average daily return: {stats['avg_daily_return_pct']:.4f}%")
print(f"Daily return std dev: {stats['daily_return_std_pct']:.4f}%")
print(f"Average volatility (annualized): {stats['avg_volatility']:.2%}")
print(f"Max drawdown: {stats['max_drawdown_pct']:.2f}%")
print(f"Best single day: {stats['max_single_day_return_pct']:.2f}%")
print(f"Worst single day: {stats['min_single_day_return_pct']:.2f}%")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{ticker} Analysis: {start_date} to {end_date}', fontsize=14, fontweight='bold')

# Plot 1: Price with moving averages
ax1 = axes[0, 0]
ax1.plot(df.index, df['adj_close'], label='Adjusted Close', linewidth=1.5)
ax1.plot(df.index, df['sma_20'], label='20-day SMA', linewidth=1, alpha=0.7)
ax1.plot(df.index, df['sma_50'], label='50-day SMA', linewidth=1, alpha=0.7)
ax1.set_ylabel('Price ($)')
ax1.set_title('Price with Moving Averages')
ax1.legend(loc='upper left', fontsize=8)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 2: Volume
ax2 = axes[0, 1]
ax2.bar(df.index, df['volume'] / 1e6, width=1, alpha=0.7, label='Daily Volume')
ax2.plot(df.index, df['volume_sma_20'] / 1e6, color='red', linewidth=1, label='20-day Avg')
ax2.set_ylabel('Volume (millions)')
ax2.set_title('Trading Volume')
ax2.legend(loc='upper right', fontsize=8)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 3: Rolling Volatility
ax3 = axes[1, 0]
ax3.plot(df.index, df['volatility'] * 100, linewidth=1.5, color='purple')
ax3.axhline(y=df['volatility'].mean() * 100, color='gray', linestyle='--', 
            label=f'Mean: {df["volatility"].mean():.1%}')
ax3.set_ylabel('Volatility (%)')
ax3.set_title(f'{volatility_window}-Day Rolling Volatility (Annualized)')
ax3.legend(loc='upper right', fontsize=8)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 4: Daily Returns Distribution
ax4 = axes[1, 1]
returns_clean = df['daily_return'].dropna()
ax4.hist(returns_clean * 100, bins=50, edgecolor='black', alpha=0.7)
ax4.axvline(x=returns_clean.mean() * 100, color='red', linestyle='--', 
            label=f'Mean: {returns_clean.mean()*100:.3f}%')
ax4.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax4.set_xlabel('Daily Return (%)')
ax4.set_ylabel('Frequency')
ax4.set_title('Daily Returns Distribution')
ax4.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Export Results with Scrapbook

We use `scrapbook` to save key outputs so they can be collected when running many notebooks at scale.

In [ ]:
# Glue individual metrics (these can be easily aggregated later)
sb.glue('ticker', ticker)
sb.glue('start_date', start_date)
sb.glue('end_date', end_date)
sb.glue('trading_days', stats['trading_days'])
sb.glue('total_return_pct', stats['total_return_pct'])
sb.glue('avg_volatility', stats['avg_volatility'])
sb.glue('max_drawdown_pct', stats['max_drawdown_pct'])
sb.glue('data_quality_ok', data_quality_ok)

# Glue the full stats dictionary for detailed access
sb.glue('stats', stats)

# Glue the processed dataframe for potential further analysis
sb.glue('prices', df, encoder='pandas')

# Glue the figure
sb.glue('analysis_plot', fig, encoder='display', display=False)

print("Results exported via scrapbook.")
print("These can be collected when running this notebook at scale with papermill.")

---

## How to Use This Template

### Manual Use
Simply change the parameter values in the parameters cell and run all cells.

### With Papermill (Programmatic)
```python
import papermill as pm

pm.execute_notebook(
    '3_stock_analysis_template.ipynb',      # input
    'output/AAPL_analysis.ipynb',           # output
    parameters={
        'ticker': 'AAPL',
        'start_date': '2024-01-01',
        'end_date': '2024-06-30',
        'db_path': 'data/data.db',
        'volatility_window': 20
    }
)
```

### Collecting Results
```python
import scrapbook as sb

# Read a single notebook
nb = sb.read_notebook('output/AAPL_analysis.ipynb')
print(nb.scraps['total_return_pct'].data)

# Read all notebooks in a folder
nbs = sb.read_notebooks('output/')
for path, scraps in nbs.notebook_scraps.items():
    print(f"{scraps['ticker'].data}: {scraps['total_return_pct'].data:.2f}%")
```